# Low level access

The low level access allows the user to add, query, and remove simulation, cache, or measurement documents 
using a designated procedure. 

When using the designated procedure, the user needs to supply the project name. 

In the following, we present the use of these procedures. 

## Adding data

First lets define some mock-up data.
Lets create 3 data from 3 distributions with different mean and std.

In [ ]:
import json
import os
import pandas
import numpy
from scipy.stats import norm
import  matplotlib.pyplot as plt 
from hera import datalayer

x = numpy.linspace(norm.ppf(0.01), norm.ppf(0.99), 100)

dataset1 = pandas.DataFrame(dict(x=x,y=norm.pdf(x,loc=0,scale=1)))
dataset2 = pandas.DataFrame(dict(x=x,y=norm.pdf(x,loc=0,scale=0.5)))
dataset3 = pandas.DataFrame(dict(x=x,y=norm.pdf(x,loc=0.5,scale=0.5)))

print(dataset1.head())

Now that we have data, we can save it. 
We would like to keep the connection between the data and the parameters that generated it. 
So that 
* dataset1 will be described by loc=0 and scale = 1
* dataset2 will be described by loc=0 and scale = 0.5
* dataset3 will be described by loc=0.5 and scale = 0.5

Therefore, we will save the loc and scale as metadata. 

First, we create the file name that we want to save to. 


In [ ]:
workingdir = os.path.join(os.path.abspath(os.getcwd()), "examples", "lowlevel")
print(f"The current file directory is {workingdir}")
os.makedirs(workingdir, exist_ok=True)

It is very important to save the **absolute path** (using os.path.abspath) and not the relative path 
because that way the loading process is independent of the location it is performed in. 

In [ ]:
dataset1File = os.path.join(workingdir,"dataset1.parquet")
dataset2File = os.path.join(workingdir,"dataset2.parquet")
dataset3File = os.path.join(workingdir,"dataset3.parquet")

Now we can save the dataset. We choose parquet for convenience, but any other format can be used. 

In [ ]:
dataset1.to_parquet(dataset1File,engine='pyarrow',compression='GZIP')
dataset2.to_parquet(dataset2File,engine='pyarrow',compression='GZIP')
dataset3.to_parquet(dataset3File,engine='pyarrow',compression='GZIP')

When we save the data to the database we need to specify the project name that the record is related to. 
we use 

In [ ]:
projectName = "ExampleProject"

in our example. 

Next, we add the documents to the database.

In [ ]:
datalayer.Measurements.addDocument(projectName=projectName,
                                   type="Distribution",
                                   dataFormat=datalayer.datatypes.PARQUET,
                                   resource=dataset1File,
                                   desc=dict(loc=0,scale=1))

datalayer.Measurements.addDocument(projectName=projectName,
                                   type="Distribution",
                                   dataFormat=datalayer.datatypes.PARQUET,
                                   resource=dataset2File,
                                   desc=dict(loc=0,scale=0.5))

datalayer.Measurements.addDocument(projectName=projectName,
                                   type="Distribution",
                                   dataFormat=datalayer.datatypes.PARQUET,
                                   resource=dataset3File,
                                   desc=dict(loc=0.5,scale=0.5))

The type of the document was chosen arbitrarily and can be any string. This string helps in future queries of the data. 
It can also be an empty string. 

The desc property includes the metadata in a JSON format. It can be any valid JSON. 

Each data is classified into one of the following categories.

- Measurements - Any acquisition of data from the 'real world'. Satellites, meteorological measurements and dispersion measurements, etc.
- Simulations  - Any output of a model. (OpenFOAM, WRF, LSM, etc).
- Cache        - Any data that is created during analysis and needed to be cached to accelerate the computations.


## Getting the  data

### Getting one record back
Now we will query the database for all the records in which loc=0 and scale=1. 

In [ ]:
List1 = datalayer.Measurements.getDocuments(projectName=projectName,loc=0,scale=1)

print(f"The number of documents obtained from the query {len(List1)} ")
item0 = List1[0]


Note that for consistency the query always returns a list. 

The description of the record that matched the query is 

In [ ]:
print("The description of dataset 1")
print(json.dumps(item0.desc, indent=4, sort_keys=True))

Now, we will extract the data. 

In [ ]:
dataset1FromDB = item0.getData().compute()

print(dataset1FromDB)

### Getting multiple records back

If the query is specified in a more general way. Lets get all the records in which loc=0

In [ ]:
List2 = datalayer.Measurements.getDocuments(projectName=projectName,loc=0)

print(f"The number of documents obtained from the query {len(List2)} ")

## Updating the data.

The hera system holds the name of the file on the disk and loads the data from it.
Therefore, if the datafile on the disk is overwitten, then the data of the record is changed

Lets multiply dataset1 by factor 2.  The file name is saved in the resource attribute. 

In [ ]:
dataset1['y'] *=2
dataset1FileName = item0.resource 
dataset1.to_parquet(dataset1FileName,engine='pyarrow',compression='GZIP')

In [ ]:
item0 = datalayer.Measurements.getDocuments(projectName=projectName,loc=0,scale=1)[0]
dataset1FromDB = item0.getData().compute()
print(dataset1FromDB)

## Updating the metadata. 

Lets assume we want to add another property to the first record. 
To so we wiill no update item0

In [ ]:
item0.desc['new_attribute'] = "some data"
item0.save()

In [ ]:
item0_fromdb = datalayer.Measurements.getDocuments(projectName=projectName,loc=0,scale=1)[0]
print(json.dumps(item0_fromdb.desc, indent=4, sort_keys=True))

## Deleting the metadata entry. 

We delete the metadata records similarly to the way we add them

The following will delete one record

In [ ]:
docdict = datalayer.Measurements.deleteDocuments(projectName=projectName,loc=0.5,scale=0.5)
print("The deleted document")
print(json.dumps(docdict[0], indent=4, sort_keys=True))

Now we can erase the file from the disk. It is saved in the resource property 

In [ ]:
import shutil 

if os.path.isfile(docdict[0]['resource']):
      os.remove(docdict[0]['resource'])
else: 
    shutil.rmtree(docdict[0]['resource'])

Now, we can delete several documents

In [ ]:
docdictList = datalayer.Measurements.deleteDocuments(projectName=projectName,loc=0)

for doc in docdictList:
    if os.path.isfile(doc['resource']):
        os.remove(doc['resource'])
    else: 
        shutil.rmtree(doc['resource'])


Using the project allows getting documents 